In [ ]:
import numpy as np
import cmath
import tkinter as tk
from tkinter import ttk, messagebox, filedialog
import os

# ==========================================
# PHẦN 1: THUẬT TOÁN CHOLESKY
# ==========================================

def format_number(val, precision=4):
    """Định dạng số thực hoặc số phức thành chuỗi hiển thị đẹp mắt"""
    if isinstance(val, complex) or np.iscomplexobj(val):
        r = np.round(val.real, precision)
        i = np.round(val.imag, precision)
        
        # Triệt tiêu sai số -0.0
        r = 0.0 if abs(r) < 1e-10 else r
        i = 0.0 if abs(i) < 1e-10 else i
        
        # Nếu phần ảo bằng 0, in ra như số thực bình thường
        if abs(i) < 1e-10:
            return f"{r:g}"
        # Nếu phần thực bằng 0, in ra phần ảo (vd: i, -i, 2.5i)
        elif abs(r) < 1e-10:
            if i == 1: return "i"
            if i == -1: return "-i"
            return f"{i:g}i"
        # Nếu có cả thực và ảo (vd: 2 + 3i, 1 - i)
        else:
            sign = "+" if i > 0 else "-"
            i_str = "" if abs(i) == 1 else f"{abs(i):g}"
            return f"{r:g} {sign} {i_str}i"
    else:
        v = np.round(val, precision)
        v = 0.0 if abs(v) < 1e-10 else v
        return f"{v:g}"

def matrix_to_latex_core(np_mat):
    """Hàm lõi tạo chuỗi bmatrix thuần sử dụng format_number"""
    latex_str = [r"\begin{bmatrix}"]
    for row in np_mat:
        row_str = " & ".join([format_number(val) for val in row])
        latex_str.append("  " + row_str + r" \\")
    latex_str.append(r"\end{bmatrix}")
    return "\n".join(latex_str)

def matrix_to_latex(np_mat):
    return "$$\n" + matrix_to_latex_core(np_mat) + "\n$$"

def run_cholesky(aug_mat, n_cols_b):
    m, total_cols = aug_mat.shape
    n_cols_a = total_cols - n_cols_b
    
    A = aug_mat[:, :n_cols_a].astype(complex)
    B = aug_mat[:, n_cols_a:].astype(complex)
    
    md = []
    md.append("# TRÌNH BÀY LỜI GIẢI CHOLESKY (HỖ TRỢ MỌI MA TRẬN)\n")
        
    md.append("### 1. Ma trận hệ số $A$ và vế phải $B$:\n")
    md.append("**Ma trận $A$:**\n" + matrix_to_latex(A) + "\n")
    md.append("**Ma trận $B$:**\n" + matrix_to_latex(B) + "\n")

    # KIỂM TRA KÍCH THƯỚC VÀ ĐỐI XỨNG AN TOÀN
    is_symmetric = False
    if m == n_cols_a:
        is_symmetric = np.allclose(A, A.T, atol=1e-10)

    # XỬ LÝ MA TRẬN KHÔNG ĐỐI XỨNG HOẶC KHÔNG VUÔNG
    if not is_symmetric:
        md.append("### 2. Chuyển đổi hệ phương trình\n")
        md.append("Nhận thấy ma trận $A$ không đối xứng hoặc không vuông. Để áp dụng phương pháp Cholesky, ta nhân cả hai vế với $A^T$ để tạo ra hệ tương đương $Mx = d$, với $M = A^T A$ và $d = A^T B$.\n")
        
        # Nhân ma trận sử dụng toán tử @ của numpy
        M = A.T @ A
        D = A.T @ B
        
        md.append("**Ma trận $M = A^T A$ (Đã vuông và đối xứng):**\n" + matrix_to_latex(M) + "\n")
        md.append("**Vế phải $d = A^T B$:**\n" + matrix_to_latex(D) + "\n")
        
        # Gán lại A và B bằng M và d để tiếp tục thuật toán
        A = M
        B = D
    else:
        md.append("### 2. Tính chất ma trận\n")
        md.append("Ma trận $A$ đối xứng ($A = A^T$), ta áp dụng trực tiếp phân tách $A = U^T U$.\n")

    # Lúc này A đã chắc chắn là ma trận vuông n_cols_a x n_cols_a
    n = n_cols_a
    U = np.zeros((n, n), dtype=complex)
    used_complex = False
    
    # 3. Phân tách A = U^T * U
    for i in range(n):
        sum_sq = sum(U[k, i]**2 for k in range(i))
        val = A[i, i] - sum_sq
        
        if abs(val) < 1e-10:
            md.append("### Lỗi: Ma trận suy biến\n")
            md.append(f"Xuất hiện $u_{{{i+1},{i+1}}} = 0$. Ma trận hệ số bị suy biến (các cột của ma trận ban đầu phụ thuộc tuyến tính).\n")
            return "\n".join(md)
            
        if val.real < -1e-10 and abs(val.imag) < 1e-10:
            used_complex = True
            
        U[i, i] = np.sqrt(val)
        
        for j in range(i + 1, n):
            sum_cross = sum(U[k, i] * U[k, j] for k in range(i))
            U[i, j] = (A[i, j] - sum_cross) / U[i, i]

    UT = U.T
    md.append("### 3. Phân tách thành ma trận tam giác:\n")
    if used_complex:
        md.append(">*Chú ý: Kích hoạt tính toán trên tập số phức do xuất hiện số âm dưới dấu căn.* \n\n")
    
    md.append("**Ma trận tam giác trên $U$:**\n" + matrix_to_latex(U) + "\n")
    md.append("**Ma trận tam giác dưới $U^T$:**\n" + matrix_to_latex(UT) + "\n")

    # 4. Giải U^T * Y = B
    Y = np.zeros_like(B, dtype=complex)
    for i in range(n):
        for col in range(n_cols_b):
            sum_y = sum(UT[i, j] * Y[j, col] for j in range(i))
            Y[i, col] = (B[i, col] - sum_y) / UT[i, i]
            
    md.append("### 4. Giải hệ thế tiến ($U^T y = d$):\n")
    md.append("Ta thu được vector trung gian $y = $:\n")
    md.append(matrix_to_latex(Y) + "\n")

    # 5. Giải U * X = Y
    X = np.zeros_like(B, dtype=complex)
    for i in range(n - 1, -1, -1):
        for col in range(n_cols_b):
            sum_x = sum(U[i, j] * X[j, col] for j in range(i + 1, n))
            X[i, col] = (Y[i, col] - sum_x) / U[i, i]

    md.append("### 5. Giải hệ thế ngược ($U x = y$):\n")
    
    if np.all(np.abs(X.imag) < 1e-10):
        md.append("*(Nhận xét: Nghiệm trả về là số thực hoàn toàn)*\n")
        
    md.append("Nghiệm cuối cùng của hệ phương trình là $x = $:\n")
    md.append(matrix_to_latex(X) + "\n")

    return "\n".join(md)


# ==========================================
# PHẦN 2: GIAO DIỆN TKINTER NHẬP TỪ FILE
# ==========================================

class MatrixApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Công cụ Giải Cholesky (Toàn Diện)")
        self.root.geometry("600x500")
        
        config_frame = ttk.LabelFrame(root, text="1. Cấu hình vế phải", padding=10)
        config_frame.pack(fill="x", padx=10, pady=5)
        
        ttk.Label(config_frame, text="Số cột của ma trận B (vế phải):").grid(row=0, column=0, padx=5)
        self.cols_b_var = tk.IntVar(value=1)
        ttk.Entry(config_frame, textvariable=self.cols_b_var, width=5).grid(row=0, column=1)
        
        input_frame = ttk.LabelFrame(root, text="2. Nhập ma trận [A | B]", padding=10)
        input_frame.pack(fill="both", expand=True, padx=10, pady=5)
        
        toolbar = ttk.Frame(input_frame)
        toolbar.pack(fill="x", pady=(0, 5))
        
        ttk.Button(toolbar, text="Mở file Notepad (.txt)", command=self.load_file).pack(side="left")
        ttk.Label(toolbar, text=" Hoặc copy/paste thẳng vào:", foreground="gray").pack(side="left", padx=10)
        
        self.text_area = tk.Text(input_frame, wrap="none", font=("Consolas", 11))
        self.text_area.pack(fill="both", expand=True)
        
        # Dữ liệu mẫu ma trận 4x3 (Không vuông) kích hoạt nhân A.T
        sample_data = "1 2 1 4\n2 1 -1 2\n3 1 2 5\n1 -1 1 -1"
        self.text_area.insert("1.0", sample_data)
        
        btn_frame = ttk.Frame(root, padding=10)
        btn_frame.pack(fill="x")
        ttk.Button(btn_frame, text="GIẢI & XUẤT FILE MARKDOWN", command=self.solve, style="Accent.TButton").pack(pady=10)

    def load_file(self):
        filepath = filedialog.askopenfilename(filetypes=[("Text Files", "*.txt"), ("All Files", "*.*")])
        if filepath:
            with open(filepath, 'r') as f:
                content = f.read()
            self.text_area.delete("1.0", tk.END)
            self.text_area.insert(tk.END, content)

    def solve(self):
        try:
            n_cols_b = self.cols_b_var.get()
            if n_cols_b <= 0:
                raise ValueError
        except:
            messagebox.showerror("Lỗi", "Số cột của B phải là số nguyên > 0!")
            return
            
        raw_text = self.text_area.get("1.0", tk.END).strip()
        if not raw_text:
            messagebox.showerror("Lỗi", "Dữ liệu ma trận đang trống!")
            return
            
        try:
            import io
            aug_mat = np.loadtxt(io.StringIO(raw_text))
            if aug_mat.ndim == 1: 
                aug_mat = aug_mat.reshape(1, -1)
        except Exception as e:
            messagebox.showerror("Lỗi Cú Pháp", "Không thể đọc ma trận.\nChi tiết: " + str(e))
            return
            
        m, total_cols = aug_mat.shape
        if total_cols <= n_cols_b:
            messagebox.showerror("Lỗi Kích Thước", f"Tổng số cột đọc được ({total_cols}) phải lớn hơn số cột của B ({n_cols_b})!")
            return
            
        save_path = filedialog.asksaveasfilename(
            defaultextension=".md",
            filetypes=[("Markdown Files", "*.md")],
            title="Lưu lời giải vào file"
        )
        
        if not save_path:
            return 
            
        print("\n[HỆ THỐNG] Đang giải và tạo file Markdown...")
        
        # BỌC LẠI BẰNG TRY-EXCEPT ĐỂ TRÁNH APP BỊ CRASH SILENTLY
        try:
            md_content = run_cholesky(aug_mat, n_cols_b)
        except Exception as e:
            messagebox.showerror("Lỗi Thuật Toán", f"Đã xảy ra lỗi nghiêm trọng trong quá trình giải:\n{str(e)}")
            return
        
        with open(save_path, 'w', encoding='utf-8') as f:
            f.write(md_content)
            
        messagebox.showinfo("Thành công", f"Đã xuất lời giải thành công ra file:\n{save_path}")

if __name__ == "__main__":
    root = tk.Tk()
    style = ttk.Style()
    style.configure("Accent.TButton", font=("Arial", 10, "bold"), foreground="black")
    app = MatrixApp(root)
    root.mainloop()


[HỆ THỐNG] Đang giải và tạo file Markdown...
